In [0]:
spark.sql("DROP TABLE IF EXISTS smart_telemetry_electricity_usage_pricing.default.realtime_meter_billing")
dbutils.fs.rm("/Volumes/smart_telemetry_electricity_usage_pricing/default/checkpoints", recurse=True)


True

In [0]:
from pyspark.sql.functions import col, from_json, date_trunc, expr, when, upper
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
import os
# ==========================================
# 1. SECURITY & CONNECTION CONFIGURATIONS
# ==========================================

EH_NAMESPACE = "entsoeelectricityusagepricing"

# Smart Meter Stream Config
METER_EH_NAME = "smart-meter-telemetry"
METER_ACCESS_KEY = os.getenv("METER_ACCESS_KEY")
METER_POLICY_NAME = "telemetry-consumer-policy"
METER_CONN_STRING = (
    f"Endpoint=sb://{EH_NAMESPACE}.servicebus.windows.net/;"
    f"SharedAccessKeyName={METER_POLICY_NAME};"
    f"SharedAccessKey={METER_ACCESS_KEY}"
)
meter_bootstrap = f"{EH_NAMESPACE}.servicebus.windows.net:9093"
meter_sasl = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{METER_CONN_STRING}";'

kafka_config_meter = {
    "kafka.bootstrap.servers": meter_bootstrap,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": meter_sasl,
    "subscribe": METER_EH_NAME,
    "startingOffsets": "earliest" 
}

# Pricing Reference Config
PRICING_EH_NAME = "day-ahead-pricing"
PRICING_ACCESS_KEY   = os.getenv("PRICING_ACCESS_KEY")
PRICING_POLICY_NAME  = "pricing-consumer-policy"
PRICING_CONN_STRING = (
    f"Endpoint=sb://{EH_NAMESPACE}.servicebus.windows.net/;"
    f"SharedAccessKeyName={PRICING_POLICY_NAME};"
    f"SharedAccessKey={PRICING_ACCESS_KEY}"
)
pricing_bootstrap = f"{EH_NAMESPACE}.servicebus.windows.net:9093"
pricing_sasl = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{PRICING_CONN_STRING}";'

kafka_config_pricing = {
    "kafka.bootstrap.servers": pricing_bootstrap,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": pricing_sasl,
    "subscribe": PRICING_EH_NAME,
    "startingOffsets": "earliest"  # Ingest all available intervals for the day
}

# ==========================================
# 2. DATA SCHEMA DEFINITIONS
# ==========================================
meter_schema = StructType([
    StructField("meter_id",        StringType(),    True),
    StructField("tariff_grid_id",  StringType(),    True),
    StructField("kwh_consumed",    DoubleType(),    True),
    StructField("voltage",         DoubleType(),    True),
    StructField("timestamp",       TimestampType(), True),
])
pricing_schema = StructType([
    StructField("timestamp", TimestampType(), True),  # Start of 15-min interval
    StructField("value", DoubleType(), True),
    StructField("price_unit",    StringType(),    True),
    StructField("currency",        StringType(),    True),
    StructField("eic_bidding_zone",        StringType(),    True),
])

# ==========================================
# 3. READ THE DAY'S PRICING LOOKUP (STATIC)
# ==========================================
# Using standard .read instead of .readStream since the day's matrix is pre-populated
df_pricing_raw = spark.read.format("kafka").options(**kafka_config_pricing).load()

df_pricing_static = (df_pricing_raw
    .select(from_json(col("value").cast("string"), pricing_schema).alias("p"))
    .select("p.*")  # Unpack struct first
    .withColumnRenamed("timestamp", "pricing_timestamp")  # Explicitly rename to avoid join ambiguity
)

# ==========================================
# 4. READ THE LIVE METER TELEMETRY (STREAM)
# ==========================================
df_meter_raw = spark.readStream.format("kafka").options(**kafka_config_meter).load()

# Parse payload and derive the active 15-minute window matching the pricing grid
df_meter_stream = (df_meter_raw
    .select(from_json(col("value").cast("string"), meter_schema).alias("data"))
    .select("data.*")
    .withColumn("pricing_interval_start", 
                date_trunc("hour", col("timestamp")) + expr("floor(minute(timestamp) / 15) * interval 15 minutes"))
)

# ==========================================
# 5. STREAM-TO-STATIC JOIN & ENRICHMENT
# ==========================================
df_realtime_cost = (df_meter_stream.join(
        df_pricing_static,
        (df_meter_stream.pricing_interval_start == df_pricing_static.pricing_timestamp) & 
        (df_meter_stream.tariff_grid_id == df_pricing_static.eic_bidding_zone),
        how="inner"
    )
    .withColumn("price_per_kwh",
                when(upper(col("price_unit")).contains("MWH"), col("value") / 1000)
                .otherwise(col("value")))
    .withColumn("calculated_cost", col("kwh_consumed") * col("price_per_kwh"))
    .select(
        col("meter_id"),
        col("kwh_consumed"),
        col("pricing_interval_start"),
        col("price_per_kwh").alias("kwh_price"),
        col("calculated_cost")
    )
)
# ==========================================
# 6. WRITE STREAM CONTINUOUSLY TO DELTA
# ==========================================
query = (df_realtime_cost.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/smart_telemetry_electricity_usage_pricing/default/checkpoints")
    .toTable("smart_telemetry_electricity_usage_pricing.default.realtime_meter_billing")
)

# Prevent premature shutdown of production clusters/notebooks
spark.streams.awaitAnyTermination()
# display(spark.sql("SHOW CATALOGS"))
#display(spark.sql("SHOW SCHEMAS IN smart_telemetry_electricity_usage_pricing"))

In [0]:
display(df_pricing_static.orderBy("pricing_timestamp"))

pricing_timestamp,value,price_unit,currency,eic_bidding_zone
2026-10-09T22:00:00.000Z,9.99,MWH,EUR,SE1
2026-10-09T22:00:00.000Z,11.16,MWH,EUR,SE3
2026-10-09T22:00:00.000Z,13.61,MWH,EUR,SE4
2026-10-09T22:00:00.000Z,10.17,MWH,EUR,SE2
2026-10-09T22:15:00.000Z,11.41,MWH,EUR,SE3
2026-10-09T22:15:00.000Z,14.38,MWH,EUR,SE4
2026-10-09T22:15:00.000Z,10.21,MWH,EUR,SE2
2026-10-09T22:30:00.000Z,16.12,MWH,EUR,SE3
2026-10-09T22:30:00.000Z,12.91,MWH,EUR,SE1
2026-10-09T22:30:00.000Z,19.22,MWH,EUR,SE4


In [0]:
display(spark.sql("""
    SELECT * FROM smart_telemetry_electricity_usage_pricing.default.realtime_meter_billing
    ORDER BY pricing_interval_start DESC
    LIMIT 100
"""))

meter_id,kwh_consumed,pricing_interval_start,kwh_price,calculated_cost
MTR-000089,0.004327,2026-10-12T01:15:00.000Z,33.94,0.14685837999999998
MTR-000011,0.004114,2026-10-12T01:15:00.000Z,27.87,0.11465718
MTR-000034,0.002981,2026-10-12T01:15:00.000Z,37.01,0.11032681
MTR-000042,0.003426,2026-10-12T01:15:00.000Z,107.23,0.36736998
MTR-000053,0.001724,2026-10-12T01:15:00.000Z,37.01,0.06380524
MTR-000027,0.004867,2026-10-12T01:15:00.000Z,33.94,0.16518597999999998
MTR-000031,0.003053,2026-10-12T01:15:00.000Z,27.87,0.08508711000000001
MTR-000035,0.00147,2026-10-12T01:15:00.000Z,107.23,0.1576281
MTR-000039,0.00382,2026-10-12T01:15:00.000Z,107.23,0.4096186
MTR-000043,0.002216,2026-10-12T01:15:00.000Z,27.87,0.06175992


In [0]:
display(spark.sql("""
    SELECT
    meter_id,
    ROUND(SUM(kwh_consumed), 3)    AS total_kwh,
    ROUND(SUM(calculated_cost), 4) AS total_cost,
    COUNT(*)                       AS readings,
    MAX(pricing_interval_start)    AS latest_interval
FROM smart_telemetry_electricity_usage_pricing.default.realtime_meter_billing
WHERE pricing_interval_start >= current_date()
  AND pricing_interval_start <  current_date() + INTERVAL 1 DAY
GROUP BY meter_id
ORDER BY total_cost DESC;
"""))

meter_id,total_kwh,total_cost,readings,latest_interval
MTR-000904,3.137,0.3131,584,2026-10-12T01:45:00.000Z
MTR-000803,3.124,0.3118,584,2026-10-12T01:45:00.000Z
MTR-001896,3.114,0.3108,584,2026-10-12T01:45:00.000Z
MTR-001438,3.096,0.309,584,2026-10-12T01:45:00.000Z
MTR-000020,3.08,0.3081,584,2026-10-12T01:45:00.000Z
MTR-000922,3.086,0.3079,584,2026-10-12T01:45:00.000Z
MTR-001256,3.081,0.3073,584,2026-10-12T01:45:00.000Z
MTR-000793,3.072,0.3068,584,2026-10-12T01:45:00.000Z
MTR-000243,3.075,0.3066,584,2026-10-12T01:45:00.000Z
MTR-001040,3.064,0.3057,584,2026-10-12T01:45:00.000Z
